<a href="https://colab.research.google.com/github/ChaitanReddy/Gen-AI-Assignment/blob/main/Task_4_BERT_Fine_Tuning_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLP Assignment – BERT Fine-Tuning

Text Classification using BERT (bert-base-uncased) with preprocessing, training, evaluation, and experiments.


In [2]:
# Install dependencies
!pip install transformers datasets scikit-learn torch

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

In [4]:
# Load dataset (IMDB)
dataset = load_dataset('imdb')
train_data = dataset['train']
test_data = dataset['test']


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [5]:
# Tokenization
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def tokenize(example):
    return tokenizer(example['text'], truncation=True, padding='max_length', max_length=128)

train_data = train_data.map(tokenize, batched=True)
test_data = test_data.map(tokenize, batched=True)

train_data.set_format(type='torch', columns=['input_ids','attention_mask','label'])
test_data.set_format(type='torch', columns=['input_ids','attention_mask','label'])


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [6]:
# Load model
model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
optimizer = AdamW(model.parameters(), lr=2e-5)


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
# Training (light version for demo)
model.train()

for epoch in range(1):
    for batch in train_data.select(range(200)):
        outputs = model(
            input_ids=batch['input_ids'].unsqueeze(0),
            attention_mask=batch['attention_mask'].unsqueeze(0),
            labels=batch['label'].unsqueeze(0)
        )
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()


In [13]:
# Evaluation
model.eval()

preds, labels = [], []

# Sample a balanced subset for evaluation
from collections import Counter

test_labels = [example['label'] for example in test_data.select(range(1000))] # Temporarily use a larger range to check distribution
label_counts = Counter(test_labels)

# Determine how many samples to take from each class for a balanced subset of 200
samples_per_class = 100 # For a total of 200 samples

balanced_test_indices = []
class_0_indices = [i for i, label in enumerate(test_labels) if label == 0]
class_1_indices = [i for i, label in enumerate(test_labels) if label == 1]

# Take `samples_per_class` from each class, ensuring we don't exceed available samples
balanced_test_indices.extend(class_0_indices[:samples_per_class])
balanced_test_indices.extend(class_1_indices[:samples_per_class])

# Shuffle the indices to mix classes
import random
random.shuffle(balanced_test_indices)


with torch.no_grad():
    # Iterate over the balanced subset
    for idx in balanced_test_indices:
        batch = test_data[idx]
        outputs = model(
            input_ids=batch['input_ids'].unsqueeze(0),
            attention_mask=batch['attention_mask'].unsqueeze(0)
        )
        logits = outputs.logits
        pred = torch.argmax(logits, dim=1).item()
        preds.append(pred)
        labels.append(batch['label'].item())


acc = accuracy_score(labels, preds)
precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')

print("Accuracy:", acc)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

print("Confusion Matrix:")
print(confusion_matrix(labels, preds))

Accuracy: 1.0
Precision: 0.0
Recall: 0.0
F1 Score: 0.0
Confusion Matrix:
[[100]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py

In [9]:
results = {}
results['Full Fine-tune (Light)'] = {
    'Accuracy': acc,
    'Precision': precision,
    'Recall': recall,
    'F1 Score': f1
}

print("Initial Full Fine-tune (Light) Results:")
for metric, value in results['Full Fine-tune (Light)'].items():
    print(f"  {metric}: {value:.4f}")

Initial Full Fine-tune (Light) Results:
  Accuracy: 1.0000
  Precision: 0.0000
  Recall: 0.0000
  F1 Score: 0.0000


### Experiment 1: Freeze BERT Layers


In [14]:
# Load a fresh model for this experiment
model_frozen = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Freeze all parameters in the BERT encoder
for param in model_frozen.bert.parameters():
    param.requires_grad = False

# Ensure the classifier layers are trainable
for param in model_frozen.classifier.parameters():
    param.requires_grad = True

optimizer_frozen = AdamW(model_frozen.parameters(), lr=2e-5)

# Training Loop (light version)
model_frozen.train()
print("\n--- Training with Frozen BERT Layers ---")
for epoch in range(1):
    for i, batch in enumerate(train_data.select(range(200))):
        optimizer_frozen.zero_grad()
        outputs = model_frozen(
            input_ids=batch['input_ids'].unsqueeze(0),
            attention_mask=batch['attention_mask'].unsqueeze(0),
            labels=batch['label'].unsqueeze(0)
        )
        loss = outputs.loss
        loss.backward()
        optimizer_frozen.step()
        if i % 50 == 0:
            print(f"  Batch {i}/{len(train_data.select(range(200)))} Loss: {loss.item():.4f}")

# Evaluation Loop
model_frozen.eval()
preds_frozen, labels_frozen = [], []

print("\n--- Evaluating Frozen BERT Layers ---")
with torch.no_grad():
    # Iterate over the balanced subset for evaluation (created above)
    for idx in balanced_test_indices:
        batch = test_data[idx]
        outputs = model_frozen(
            input_ids=batch['input_ids'].unsqueeze(0),
            attention_mask=batch['attention_mask'].unsqueeze(0)
        )
        logits = outputs.logits
        pred = torch.argmax(logits, dim=1).item()
        preds_frozen.append(pred)
        labels_frozen.append(batch['label'].item())

acc_frozen = accuracy_score(labels_frozen, preds_frozen)
precision_frozen, recall_frozen, f1_frozen, _ = precision_recall_fscore_support(labels_frozen, preds_frozen, average='binary', zero_division=0)

results['Frozen BERT Layers'] = {
    'Accuracy': acc_frozen,
    'Precision': precision_frozen,
    'Recall': recall_frozen,
    'F1 Score': f1_frozen
}

print("\nFrozen BERT Layers Results:")
for metric, value in results['Frozen BERT Layers'].items():
    print(f"  {metric}: {value:.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



--- Training with Frozen BERT Layers ---
  Batch 0/200 Loss: 0.7070
  Batch 50/200 Loss: 0.4529
  Batch 100/200 Loss: 0.2507
  Batch 150/200 Loss: 0.2622

--- Evaluating Frozen BERT Layers ---

Frozen BERT Layers Results:
  Accuracy: 0.9900
  Precision: 0.0000
  Recall: 0.0000
  F1 Score: 0.0000


### Experiment 2: Fine-tune Last 2 Layers



In [15]:
# Load a fresh model for this experiment
model_last_2 = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Freeze all parameters in the BERT encoder initially
for param in model_last_2.bert.parameters():
    param.requires_grad = False

# Unfreeze the last two encoder layers
# BERT's encoder layers are typically accessed via model.bert.encoder.layer
# Python list indexing allows access to the last elements using negative indices
for i in range(1, 3): # -1 and -2 for the last two layers
    for param in model_last_2.bert.encoder.layer[-i].parameters():
        param.requires_grad = True

# Ensure the classifier layers are trainable
for param in model_last_2.classifier.parameters():
    param.requires_grad = True

optimizer_last_2 = AdamW(model_last_2.parameters(), lr=2e-5)

# Training Loop (light version)
model_last_2.train()
print("\n--- Training with Fine-tuned Last 2 BERT Layers ---")
for epoch in range(1):
    for i, batch in enumerate(train_data.select(range(200))):
        optimizer_last_2.zero_grad()
        outputs = model_last_2(
            input_ids=batch['input_ids'].unsqueeze(0),
            attention_mask=batch['attention_mask'].unsqueeze(0),
            labels=batch['label'].unsqueeze(0)
        )
        loss = outputs.loss
        loss.backward()
        optimizer_last_2.step()
        if i % 50 == 0:
            print(f"  Batch {i}/{len(train_data.select(range(200)))} Loss: {loss.item():.4f}")

# Evaluation Loop
model_last_2.eval()
preds_last_2, labels_last_2 = [], []

print("\n--- Evaluating Fine-tuned Last 2 BERT Layers ---")
with torch.no_grad():
    # Iterate over the balanced subset for evaluation (created above)
    for idx in balanced_test_indices:
        batch = test_data[idx]
        outputs = model_last_2(
            input_ids=batch['input_ids'].unsqueeze(0),
            attention_mask=batch['attention_mask'].unsqueeze(0)
        )
        logits = outputs.logits
        pred = torch.argmax(logits, dim=1).item()
        preds_last_2.append(pred)
        labels_last_2.append(batch['label'].item())

acc_last_2 = accuracy_score(labels_last_2, preds_last_2)
precision_last_2, recall_last_2, f1_last_2, _ = precision_recall_fscore_support(labels_last_2, preds_last_2, average='binary', zero_division=0)

results['Fine-tune Last 2 Layers'] = {
    'Accuracy': acc_last_2,
    'Precision': precision_last_2,
    'Recall': recall_last_2,
    'F1 Score': f1_last_2
}

print("\nFine-tune Last 2 Layers Results:")
for metric, value in results['Fine-tune Last 2 Layers'].items():
    print(f"  {metric}: {value:.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



--- Training with Fine-tuned Last 2 BERT Layers ---
  Batch 0/200 Loss: 0.4365
  Batch 50/200 Loss: 0.0739
  Batch 100/200 Loss: 0.0499
  Batch 150/200 Loss: 0.0237

--- Evaluating Fine-tuned Last 2 BERT Layers ---

Fine-tune Last 2 Layers Results:
  Accuracy: 1.0000
  Precision: 0.0000
  Recall: 0.0000
  F1 Score: 0.0000


### Experiment 3: Compare Performance Metrics



In [12]:
import pandas as pd

results_df = pd.DataFrame(results).T
print("\n--- Comparison of Experiment Results ---")
display(results_df.round(4))


--- Comparison of Experiment Results ---


,Accuracy,Precision,Recall,F1 Score
Full Fine-tune (Light),1.0,0.0,0.0,0.0
Frozen BERT Layers,1.0,0.0,0.0,0.0
Fine-tune Last 2 Layers,1.0,0.0,0.0,0.0
